# Docling PDF 파서 예제

PDF에서 **텍스트(text)**, **테이블(table)**, **이미지(image)** 를 분리 추출하는 예제입니다.  
이미지는 `DocumentFigureClassifier` 모델로 **16가지 카테고리**로 자동 분류됩니다.  
모든 요소에서 **페이지 번호**와 **바운딩 박스(bounding box)** 위치 정보를 함께 추출합니다.

### 분류 카테고리 (16종)
`bar_chart`, `bar_code`, `chemistry_markush_structure`, `chemistry_molecular_structure`,  
`flow_chart`, `icon`, `line_chart`, `logo`, `map`, `other`,  
`pie_chart`, `qr_code`, `remote_sensing`, `screenshot`, `signature`, `stamp`

## 설치
```bash
pip install docling pandas pillow tabulate
```

> **맥북 실리콘(Apple Silicon) 참고**: `DocumentFigureClassifier`는 EfficientNet-B0 기반으로 CPU에서도 빠르게 동작합니다.  
> NVIDIA GPU 없이 바로 사용 가능합니다.

## 1. 라이브러리 임포트 및 설정

In [ ]:
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import Image as IPImage, display

from docling_core.types.doc import PictureItem, TableItem
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TableFormerMode,
    TableStructureOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption

# 출력 디렉토리 설정
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# 하위 디렉토리 미리 생성
BBOX_DIR        = OUTPUT_DIR / "bbox"          # bounding box 시각화 이미지
TABLE_IMG_DIR   = OUTPUT_DIR / "table" / "img" # 테이블 영역 이미지
TABLE_MD_DIR    = OUTPUT_DIR / "table" / "md"  # 테이블 마크다운

for d in (BBOX_DIR, TABLE_IMG_DIR, TABLE_MD_DIR):
    d.mkdir(parents=True, exist_ok=True)

# 이미지 해상도 스케일 (1.0 = 72 DPI, 2.0 = 144 DPI)
IMAGE_SCALE = 2.0

## 2. PDF 변환 (Conversion)

파이프라인 옵션(pipeline options)에서 이미지 생성 및 그림 분류를 활성화합니다.

**TableFormer 테이블 추출 옵션:**
- `do_table_structure=True` (기본값): TableFormer 모델로 셀 구조 분석
- `TableFormerMode.ACCURATE`: 복잡한 병합 셀/다단 헤더에 권장 (느리지만 정확)
- `TableFormerMode.FAST`: 단순 테이블 또는 대량 처리 시 사용
- `do_cell_matching=True` (기본값): 구조 예측과 실제 셀 텍스트를 매칭
- `generate_table_images=True`: 테이블 영역 이미지를 별도 파일로 저장

In [ ]:
# 분석할 PDF 경로 설정 (원하는 PDF 경로로 변경하세요)
PDF_PATH = Path("sample.pdf")

# PDF가 없을 경우 샘플 URL로 다운로드
if not PDF_PATH.exists():
    import urllib.request
    sample_url = "https://arxiv.org/pdf/2206.01062"
    print(f"샘플 PDF 다운로드 중: {sample_url}")
    urllib.request.urlretrieve(sample_url, PDF_PATH)
    print("다운로드 완료")

# 파이프라인 옵션 설정
# - generate_page_images: 페이지 전체 이미지 생성 (bounding box 시각화에 필요)
# - generate_picture_images: 그림(figure) 이미지 생성
# - generate_table_images: 테이블 영역 이미지 생성
# - do_picture_classification: DocumentFigureClassifier로 그림 종류 자동 분류
# - do_table_structure: TableFormer 모델로 테이블 셀 구조 분석 (기본값 True)
# - table_structure_options:
#     mode=ACCURATE: 병합 셀/다단 헤더 등 복잡한 테이블에 권장
#     mode=FAST: 단순 테이블 또는 대량 처리 시 속도 우선
#     do_cell_matching=True: 구조 예측과 실제 셀 텍스트를 매칭 (기본값 True)
pipeline_options = PdfPipelineOptions()
pipeline_options.images_scale = IMAGE_SCALE
pipeline_options.generate_page_images = True
pipeline_options.generate_picture_images = True
pipeline_options.generate_table_images = True   # 테이블 영역 이미지 저장
pipeline_options.do_picture_classification = True
pipeline_options.do_table_structure = True       # 기본값이지만 명시적으로 설정
pipeline_options.table_structure_options = TableStructureOptions(
    mode=TableFormerMode.ACCURATE,  # FAST로 변경하면 속도 우선
    do_cell_matching=True,
)

# 문서 변환기(converter) 초기화
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

print("PDF 변환 중... (처음 실행 시 모델 다운로드로 시간이 걸릴 수 있습니다)")
result = converter.convert(PDF_PATH)
doc = result.document
doc_name = result.input.file.stem
print(f"변환 완료: {doc_name}")

## 3. 위치 정보(Bounding Box) 헬퍼 함수

Docling의 위치 정보 구조:
- `element.prov` → `list[ProvenanceItem]` (보통 prov[0] 사용)
- `prov.page_no` → 1-based 페이지 번호
- `prov.bbox` → `BoundingBox(l, t, r, b)` — 기본 좌표 원점은 **BOTTOMLEFT** (PDF 표준)
- `doc.pages[page_no].size` → 페이지 크기 `Size(width, height)` (포인트 단위, 1pt = 1/72 inch)
- `bbox.to_top_left_origin(page_height)` → 화면 좌표계(좌상단 원점)로 변환

In [ ]:
def get_location(element, doc) -> dict | None:
    """element의 위치 정보를 딕셔너리로 반환.

    반환값:
        page_no  : 페이지 번호 (1-based)
        page_w/h : 페이지 크기 (포인트 단위)
        bbox_raw : 원본 BoundingBox (BOTTOMLEFT 원점, PDF 표준)
        bbox_tl  : 좌상단 원점으로 변환한 BoundingBox (화면/이미지 좌표계)
    """
    if not element.prov:
        return None
    p = element.prov[0]
    page = doc.pages.get(p.page_no)
    if page is None:
        return None
    # PDF 기본 좌표계(BOTTOMLEFT)를 화면 좌표계(TOPLEFT)로 변환
    bbox_tl = p.bbox.to_top_left_origin(page.size.height)
    return {
        "page_no": p.page_no,
        "page_w": page.size.width,
        "page_h": page.size.height,
        # 원본 좌표 (BOTTOMLEFT): PDF 표준 좌표계 — 좌하단이 원점
        "bbox_raw": {"l": p.bbox.l, "t": p.bbox.t, "r": p.bbox.r, "b": p.bbox.b},
        # 변환 좌표 (TOPLEFT): 화면/이미지 좌표계 — 좌상단이 원점 (일반적으로 더 직관적)
        "bbox_tl": {"l": bbox_tl.l, "t": bbox_tl.t, "r": bbox_tl.r, "b": bbox_tl.b},
    }

## 4. 텍스트(Text) 추출

In [ ]:
# 전체 텍스트를 마크다운(Markdown) 형식으로 추출
markdown_text = doc.export_to_markdown()

# 텍스트 파일로 저장
text_path = OUTPUT_DIR / f"{doc_name}_text.md"
text_path.write_text(markdown_text, encoding="utf-8")
print(f"텍스트 저장 완료: {text_path}")

# 앞부분 미리보기
print("\n--- 텍스트 미리보기 (앞 500자) ---")
print(markdown_text[:500])

## 5. 테이블(Table) 추출 + 위치 정보

### TableFormer 추출 결과 구조

```
TableItem
├── data.grid          : list[list[TableCell]]  — 2D 셀 배열 [row][col]
│   └── TableCell
│       ├── text       : str       — 셀 텍스트
│       ├── row_span   : int       — 병합된 행 수 (1 = 병합 없음)
│       ├── col_span   : int       — 병합된 열 수
│       ├── start_row_offset_idx / end_row_offset_idx
│       ├── start_col_offset_idx / end_col_offset_idx
│       └── column_header : bool  — 헤더 셀 여부
├── export_to_dataframe()  → pandas DataFrame
├── export_to_html()       → HTML 문자열
└── export_to_markdown()   → Markdown 문자열
```

In [ ]:
tables = doc.tables
print(f"감지된 테이블 수: {len(tables)}개")

for i, table in enumerate(tables):
    # --- 위치 정보(bounding box) 추출 ---
    loc = get_location(table, doc)
    if loc:
        bb = loc["bbox_tl"]  # 좌상단 원점 기준 좌표
        print(f"\n=== 테이블 {i+1} | 페이지 {loc['page_no']} "
              f"| bbox(좌상단 기준) l={bb['l']:.1f} t={bb['t']:.1f} "
              f"r={bb['r']:.1f} b={bb['b']:.1f} "
              f"| 페이지 크기 {loc['page_w']:.0f}x{loc['page_h']:.0f}pt ===")
    else:
        print(f"\n=== 테이블 {i+1} | 위치 정보 없음 ===")

    # --- 셀 구조 분석 (TableFormer 결과) ---
    grid = table.data.grid  # list[list[TableCell]]
    n_rows, n_cols = len(grid), len(grid[0]) if grid else 0
    print(f"    크기: {n_rows}행 × {n_cols}열")

    # 병합 셀(row_span > 1 또는 col_span > 1) 개수 집계
    merged = sum(
        1 for row in grid for cell in row
        if cell.row_span > 1 or cell.col_span > 1
    )
    # 헤더 행 여부 확인 (column_header=True인 셀이 있는 행)
    header_rows = {
        cell.start_row_offset_idx
        for row in grid for cell in row
        if cell.column_header
    }
    print(f"    병합 셀: {merged}개 | 헤더 행: {sorted(header_rows)}")

    # pandas DataFrame으로 변환
    df: pd.DataFrame = table.export_to_dataframe(doc=doc)
    display(df)

    # 마크다운으로 저장 (table/md 폴더)
    md_path = TABLE_MD_DIR / f"{doc_name}_table_{i+1}.md"
    md_path.write_text(table.export_to_markdown(doc=doc), encoding="utf-8")

    # 테이블 이미지 저장 (table/img 폴더)
    table_img = table.get_image(doc)
    if table_img:
        img_path = TABLE_IMG_DIR / f"{doc_name}_table_{i+1}.png"
        with img_path.open("wb") as f:
            table_img.save(f, "PNG")
        print(f"    이미지: table/img/{img_path.name}")

    print(f"    저장: table/md/{md_path.name}")

## 6. 이미지(Image) 추출, 분류 + 위치 정보

In [ ]:
picture_count = 0
table_img_count = 0
# 카테고리별 그림 목록 집계
category_summary: dict[str, list[str]] = defaultdict(list)

for element, _level in doc.iterate_items():
    # 그림(figure) 이미지 추출 + 분류 + 위치 정보
    if isinstance(element, PictureItem):
        picture_count += 1

        # --- 분류 결과(classification) 읽기 ---
        # PictureClassificationData.predicted_classes 리스트에서 최고 신뢰도 클래스 선택
        category = "unknown"
        confidence = 0.0
        for ann in element.get_annotations():
            if hasattr(ann, "predicted_classes") and ann.predicted_classes:
                # predicted_classes는 신뢰도 내림차순 정렬되어 있음
                top = ann.predicted_classes[0]
                category = top.class_name
                confidence = top.confidence
                break

        # --- 위치 정보(bounding box) 추출 ---
        loc = get_location(element, doc)
        loc_str = ""
        if loc:
            bb = loc["bbox_tl"]  # 좌상단 원점 기준 좌표
            loc_str = (f" | 페이지 {loc['page_no']} "
                       f"bbox l={bb['l']:.1f} t={bb['t']:.1f} "
                       f"r={bb['r']:.1f} b={bb['b']:.1f}")

        # 카테고리별 하위 디렉토리에 저장
        cat_dir = OUTPUT_DIR / "pictures" / category
        cat_dir.mkdir(parents=True, exist_ok=True)
        img_path = cat_dir / f"{doc_name}_picture_{picture_count}.png"

        if category == "logo":
            continue  # logo는 쓰레기 정보 제외
        with img_path.open("wb") as f:
            element.get_image(doc).save(f, "PNG")

        category_summary[category].append(img_path.name)
        print(f"[{category:35s}] conf={confidence:.2f}{loc_str}")
        display(IPImage(filename=str(img_path), width=400))

    # 테이블 이미지 추출 + 위치 정보
    if isinstance(element, TableItem):
        table_img_count += 1
        loc = get_location(element, doc)
        loc_str = ""
        if loc:
            bb = loc["bbox_tl"]
            loc_str = (f" | 페이지 {loc['page_no']} "
                       f"bbox l={bb['l']:.1f} t={bb['t']:.1f} "
                       f"r={bb['r']:.1f} b={bb['b']:.1f}")
        # img_path = OUTPUT_DIR / f"{doc_name}_table_img_{table_img_count}.png"
        # with img_path.open("wb") as f:
        #     element.get_image(doc).save(f, "PNG")
        print(f"[테이블 이미지 {table_img_count:3d}]{loc_str}")

print(f"\n총 그림: {picture_count}개, 테이블 이미지: {table_img_count}개")
print("\n--- 카테고리별 분류 결과 ---")
for cat, files in sorted(category_summary.items()):
    print(f"  {cat:35s}: {len(files)}개")

## 7. Bounding Box 시각화 검증

추출한 위치 정보가 실제 PDF 페이지에서 올바른지 OpenCV로 시각화합니다.

**좌표 변환 원리:**
- `bbox_tl` 좌표는 포인트(pt) 단위, 페이지 크기 기준
- 페이지 이미지는 `IMAGE_SCALE` 배율로 렌더링 → 픽셀 좌표 = pt 좌표 × `IMAGE_SCALE`

In [ ]:
import cv2
import numpy as np
from PIL import Image as PILImage

# 색상 정의 (BGR)
COLOR_PICTURE = (0, 200, 0)   # 초록 — 그림(figure)
COLOR_TABLE   = (0, 0, 220)   # 빨강 — 테이블


def draw_bboxes_on_page(doc, page_no: int, elements: list, scale: float = IMAGE_SCALE):
    """특정 페이지 이미지에 여러 element의 bounding box를 그려서 반환.

    Args:
        doc      : Docling 변환 결과 document
        page_no  : 시각화할 페이지 번호 (1-based)
        elements : (element, label, color) 튜플 리스트
        scale    : 사용하지 않음 (하위 호환용 파라미터, 실제 이미지 크기로 자동 계산)

    Returns:
        OpenCV BGR 이미지 (numpy array)

    좌표 변환 원리:
        bbox_tl 좌표는 pt 단위, page.size 도 pt 단위.
        실제 픽셀 스케일 = 이미지 실제 픽셀 크기 / page.size(pt)
        IMAGE_SCALE 상수를 직접 곱하면 Docling 내부 반올림 오차로 얼라인이 틀어지므로
        반드시 실제 이미지 픽셀 크기 기준으로 스케일을 계산해야 함.
    """
    # Docling이 생성한 페이지 이미지(PIL) → OpenCV BGR 배열로 변환
    page_img_pil: PILImage.Image = doc.pages[page_no].image.pil_image
    img = cv2.cvtColor(np.array(page_img_pil), cv2.COLOR_RGB2BGR)
    img_w, img_h = page_img_pil.size  # 실제 픽셀 크기

    # 페이지 pt 크기 → 실제 픽셀 스케일 계산 (x/y 축 각각)
    page = doc.pages[page_no]
    scale_x = img_w / page.size.width
    scale_y = img_h / page.size.height

    for element, label, color in elements:
        loc = get_location(element, doc)
        if loc is None or loc["page_no"] != page_no:
            continue
        bb = loc["bbox_tl"]  # 좌상단 원점 기준 pt 좌표

        # pt → 픽셀 변환: x/y 스케일 각각 적용
        x1, y1 = int(bb["l"] * scale_x), int(bb["t"] * scale_y)
        x2, y2 = int(bb["r"] * scale_x), int(bb["b"] * scale_y)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness=2)
        # 라벨 텍스트 (박스 위쪽에 배경 포함하여 가독성 확보)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(img, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
        cv2.putText(img, label, (x1 + 2, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)

    return img


print("시각화 함수 정의 완료")

In [ ]:
# 페이지별로 element 수집
page_elements: dict[int, list] = defaultdict(list)

for element, _level in doc.iterate_items():
    if isinstance(element, PictureItem):
        # 분류 카테고리를 라벨로 사용
        label = "figure"
        for ann in element.get_annotations():
            if hasattr(ann, "predicted_classes") and ann.predicted_classes:
                label = ann.predicted_classes[0].class_name[:12]  # 너무 길면 잘라냄
                break
        loc = get_location(element, doc)
        if loc:
            page_elements[loc["page_no"]].append((element, label, COLOR_PICTURE))

    elif isinstance(element, TableItem):
        loc = get_location(element, doc)
        if loc:
            page_elements[loc["page_no"]].append((element, "table", COLOR_TABLE))

# 각 페이지 시각화 및 저장
for page_no, elems in sorted(page_elements.items()):
    img = draw_bboxes_on_page(doc, page_no, elems)

    # 파일 저장 (bbox 폴더)
    out_path = BBOX_DIR / f"{doc_name}_page{page_no:03d}_bbox.jpg"
    cv2.imwrite(str(out_path), img, [cv2.IMWRITE_JPEG_QUALITY, 90])

    # 노트북에서 인라인 표시 (PIL로 변환 후 표시)
    print(f"\n--- 페이지 {page_no} ({len(elems)}개 요소) ---")
    display(IPImage(data=cv2.imencode(".jpg", img)[1].tobytes(), width=700))

print("\n시각화 완료")

## 8.5 페이지/이미지 요약 생성 (Bedrock 멀티모달 LLM)

각 페이지 이미지와 figure 이미지를 Bedrock Claude 모델에 전달하여 요약 + 엔티티를 추출합니다.  
결과는 HTML 테이블 형태로 최종 마크다운에 삽입됩니다.

In [ ]:
import boto3
import json
import base64
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

bedrock_runtime = boto3.client("bedrock-runtime")

# 모델 ID (필요에 따라 변경)
MODEL_ID = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"

SUMMARY_PROMPT = """Analyze this image and respond ONLY with the JSON below. No other text.
{
  "summary": "3-5 sentence summary of key content (in Korean)",
  "entities": ["entity1", "entity2", ...]
}
For entities, extract key named entities such as person names, organization names, product names, technical terms, and numerical metrics.
The page-level summary context may be provided below. Use it to produce a more accurate summary and entity extraction.
IMPORTANT: Write the summary value in Korean."""

TABLE_SUMMARY_PROMPT = """Analyze this table image and respond ONLY with the JSON below. No other text.
{
  "summary": "2-3 sentence summary of the table content (in Korean)",
  "entities": ["entity1", "entity2", ...],
  "category": "table_category"
}
For entities, extract key named entities such as person names, organization names, product names, technical terms, and numerical metrics from the table.
For category, choose exactly one from: financial_statement, comparison, statistics, performance_metrics, configuration, schedule, pricing, inventory, survey_results, reference, other
The page-level summary context may be provided below. Use it to produce a more accurate summary and entity extraction.
IMPORTANT: Write the summary value in Korean."""


def call_bedrock_vision(img_pil, prompt=SUMMARY_PROMPT, model_id=MODEL_ID):
    """PIL 이미지를 Bedrock 멀티모달 LLM으로 분석하여 JSON 반환."""
    buf = BytesIO()
    img_pil.save(buf, format="PNG")
    img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    response = bedrock_runtime.invoke_model(
        modelId=model_id,
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 1024,
            "messages": [{
                "role": "user",
                "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": img_b64}},
                    {"type": "text", "text": prompt},
                ],
            }],
        }),
    )
    text = json.loads(response["body"].read())["content"][0]["text"]
    # JSON 블록 추출 (```json ... ``` 감싸기 대응)
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(text)


# --- 페이지 요약 병렬 생성 ---
page_summaries: dict[int, dict] = {}  # {page_no: {"summary": ..., "entities": [...]}}
total_pages = len(doc.pages)

def _summarize_page(page_no, page):
    return page_no, call_bedrock_vision(page.image.pil_image)

print(f"페이지 요약 병렬 생성 중... ({total_pages}페이지)")
with ThreadPoolExecutor(max_workers=min(total_pages, 10)) as executor:
    futures = {executor.submit(_summarize_page, pn, pg): pn for pn, pg in doc.pages.items()}
    for future in as_completed(futures):
        pn = futures[future]
        try:
            page_no, result = future.result()
            page_summaries[page_no] = result
            print(f"  페이지 {page_no}/{total_pages} 완료")
        except Exception as e:
            page_summaries[pn] = {"summary": f"요약 생성 실패: {e}", "entities": []}
            print(f"  페이지 {pn}/{total_pages} 오류: {e}")

# --- 이미지(figure) 요약 병렬 생성 ---
fig_elements = []
for element, _level in doc.iterate_items():
    if isinstance(element, PictureItem):
        fig_elements.append(element)

image_summaries: dict[int, dict] = {}  # {fig_idx(1-based): {"summary": ..., "entities": [...]}}

def _get_page_context(element):
    """element가 속한 페이지의 요약을 컨텍스트 문자열로 반환."""
    loc = get_location(element, doc)
    if not loc:
        return ""
    info = page_summaries.get(loc["page_no"], {})
    s = info.get("summary", "") if isinstance(info, dict) else ""
    return f"\n\n[페이지 {loc['page_no']} 컨텍스트] {s}" if s else ""

def _summarize_figure(fig_idx, element):
    img = element.get_image(doc)
    if img is None:
        return fig_idx, {"summary": "이미지 없음", "entities": []}
    ctx = _get_page_context(element)
    return fig_idx, call_bedrock_vision(img, prompt=SUMMARY_PROMPT + ctx)

print(f"\n이미지 요약 병렬 생성 중... ({len(fig_elements)}개)")
with ThreadPoolExecutor(max_workers=min(len(fig_elements) or 1, 10)) as executor:
    futures = {executor.submit(_summarize_figure, i+1, el): i+1 for i, el in enumerate(fig_elements)}
    for future in as_completed(futures):
        idx = futures[future]
        try:
            fig_idx, result = future.result()
            image_summaries[fig_idx] = result
            print(f"  이미지 {fig_idx}/{len(fig_elements)} 완료")
        except Exception as e:
            image_summaries[idx] = {"summary": f"요약 생성 실패: {e}", "entities": []}
            print(f"  이미지 {idx}/{len(fig_elements)} 오류: {e}")

# --- 테이블 요약 병렬 생성 ---
tbl_elements_for_summary = [e for e, _ in doc.iterate_items() if isinstance(e, TableItem)]
table_summaries: dict[int, dict] = {}  # {tbl_idx(1-based): {"summary": ..., "entities": [...], "category": ...}}

def _summarize_table(tbl_idx, element):
    img = element.get_image(doc)
    if img is None:
        return tbl_idx, {"summary": "이미지 없음", "entities": [], "category": "other"}
    ctx = _get_page_context(element)
    return tbl_idx, call_bedrock_vision(img, prompt=TABLE_SUMMARY_PROMPT + ctx)

print(f"\n테이블 요약 병렬 생성 중... ({len(tbl_elements_for_summary)}개)")
with ThreadPoolExecutor(max_workers=min(len(tbl_elements_for_summary) or 1, 10)) as executor:
    futures = {executor.submit(_summarize_table, i+1, el): i+1 for i, el in enumerate(tbl_elements_for_summary)}
    for future in as_completed(futures):
        idx = futures[future]
        try:
            tbl_idx_r, result = future.result()
            table_summaries[tbl_idx_r] = result
            print(f"  테이블 {tbl_idx_r}/{len(tbl_elements_for_summary)} 완료")
        except Exception as e:
            table_summaries[idx] = {"summary": f"요약 생성 실패: {e}", "entities": [], "category": "other"}
            print(f"  테이블 {idx}/{len(tbl_elements_for_summary)} 오류: {e}")

print(f"\n완료: 페이지 {len(page_summaries)}개, 이미지 {len(image_summaries)}개, 테이블 {len(table_summaries)}개")

## 9. 최종 통합 마크다운 생성

섹션 5·6에서 저장한 테이블 md와 이미지를 `<!-- image -->` placeholder 위치에 순서대로 치환합니다.  
페이지 메타데이터와 이미지 메타데이터는 HTML 테이블로 삽입됩니다.

In [ ]:
from docling_core.types.doc import PictureItem, TableItem
import re

PAGE_BREAK = "<!-- page-break -->"
final_md = doc.export_to_markdown(
    image_placeholder="<!-- image -->",
    page_break_placeholder=PAGE_BREAK,
    escape_html=False,
)

def bbox_str_for(element, doc):
    loc = get_location(element, doc)
    if not loc:
        return "", ""
    bb = loc["bbox_tl"]
    return str(loc["page_no"]), f'l={bb["l"]:.1f} t={bb["t"]:.1f} r={bb["r"]:.1f} b={bb["b"]:.1f}'

# 1단계: <!-- image --> → figure HTML 테이블 치환
pic_idx = 0
for element, _level in doc.iterate_items():
    if not isinstance(element, PictureItem):
        continue
    pic_idx += 1
    category = "unknown"
    for ann in element.get_annotations():
        if hasattr(ann, "predicted_classes") and ann.predicted_classes:
            category = ann.predicted_classes[0].class_name
            break
    img_path = OUTPUT_DIR / "pictures" / category / f"{doc_name}_picture_{pic_idx}.png"
    loc = get_location(element, doc)
    bbox_str = ""
    page_no = ""
    if loc:
        bb = loc["bbox_tl"]
        page_no = str(loc["page_no"])
        bbox_str = f'l={bb["l"]:.1f} t={bb["t"]:.1f} r={bb["r"]:.1f} b={bb["b"]:.1f}'
    fig_info = image_summaries.get(pic_idx, {"summary": "", "entities": []})
    img_src = f'![figure-{pic_idx:03d}]({img_path.relative_to(OUTPUT_DIR)})' if img_path.exists() else ''
    repl = (
        f'<table class="figure-meta">\n'
        f'  <tr>\n'
        f'    <td>image_id</td>\n'
        f'    <td>figure-{pic_idx:03d}</td>\n'
        f'  </tr>\n'
        f'  <tr>\n'
        f'    <td>category</td>\n'
        f'    <td>{category}</td>\n'
        f'  </tr>\n'
        f'  <tr>\n'
        f'    <td>page_number</td>\n'
        f'    <td>{page_no}</td>\n'
        f'  </tr>\n'
        f'  <tr>\n'
        f'    <td>image_summary</td>\n'
        f'    <td>{fig_info["summary"]}</td>\n'
        f'  </tr>\n'
        f'  <tr>\n'
        f'    <td>entities</td>\n'
        f'    <td>{", ".join(fig_info.get("entities", []))}</td>\n'
        f'  </tr>\n'
        f'  <tr>\n'
        f'    <td>bbox</td>\n'
        f'    <td>{bbox_str}</td>\n'
        f'  </tr>\n'
        f'  <tr>\n'
        f'    <td>img_source</td>\n'
        f'    <td>{img_path.relative_to(OUTPUT_DIR) if img_path.exists() else ""}</td>\n'
        f'  </tr>\n'
        f'</table>\n\n'
        f'{img_src}'
    )
    final_md = final_md.replace("<!-- image -->", repl, 1)

# 2단계: 마크다운 표 블록 → table HTML 메타데이터 테이블로 감싸기
tbl_elements = [e for e, _ in doc.iterate_items() if isinstance(e, TableItem)]
tbl_idx = 0
def wrap_table(m):
    global tbl_idx
    if tbl_idx >= len(tbl_elements):
        return m.group(0)
    element = tbl_elements[tbl_idx]
    tbl_idx += 1
    tag = f"table-{tbl_idx:03d}"
    img_path = TABLE_IMG_DIR / f"{doc_name}_table_{tbl_idx}.png"
    img_line = f"![{tag}]({img_path.relative_to(OUTPUT_DIR)})" if img_path.exists() else ""
    pn, bb = bbox_str_for(element, doc)
    tbl_info = table_summaries.get(tbl_idx, {"summary": "", "entities": [], "category": "other"})
    tbl_summary = tbl_info.get("summary", "")
    tbl_entities = ", ".join(tbl_info.get("entities", []))
    tbl_category = tbl_info.get("category", "other")
    meta = (
        f'<table class="table-meta">\n'
        f'  <tr>\n'
        f'    <td>table_id</td>\n'
        f'    <td>{tag}</td>\n'
        f'  </tr>\n'
        f'  <tr>\n'
        f'    <td>category</td>\n'
        f'    <td>{tbl_category}</td>\n'
        f'  </tr>\n'
        f'  <tr>\n'
        f'    <td>page_number</td>\n'
        f'    <td>{pn}</td>\n'
        f'  </tr>\n'
        f'  <tr>\n'
        f'    <td>table_summary</td>\n'
        f'    <td>{tbl_summary}</td>\n'
        f'  </tr>\n'
        f'  <tr>\n'
        f'    <td>entities</td>\n'
        f'    <td>{tbl_entities}</td>\n'
        f'  </tr>\n'
        f'  <tr>\n'
        f'    <td>bbox</td>\n'
        f'    <td>{bb}</td>\n'
        f'  </tr>\n'
        f'  <tr>\n'
        f'    <td>img_source</td>\n'
        f'    <td>{img_path.relative_to(OUTPUT_DIR) if img_path.exists() else ""}</td>\n'
        f'  </tr>\n'
        f'</table>\n\n'
        f'{img_line}'
    )
    return f"{meta}\n{m.group(0).strip()}"

final_md = re.sub(r"(?:^\|.+\n)+", wrap_table, final_md, flags=re.MULTILINE)

# 3단계: page 태그로 감싸기 (페이지 요약 HTML 테이블 포함)
pages = final_md.split(PAGE_BREAK)
wrapped_pages = []
for i, page in enumerate(pages):
    if not page.strip():
        continue
    page_no = i + 1
    info = page_summaries.get(page_no, {"summary": "", "entities": []})
    summary = info.get("summary", "") if isinstance(info, dict) else str(info)
    entities = ", ".join(info.get("entities", [])) if isinstance(info, dict) else ""
    page_meta = (
        f'<table class="page-meta">\n'
        f'  <tr>\n'
        f'    <td>page_number</td>\n'
        f'    <td>{page_no}</td>\n'
        f'  </tr>\n'
        f'  <tr>\n'
        f'    <td>page_summary</td>\n'
        f'    <td>{summary}</td>\n'
        f'  </tr>\n'
        f'  <tr>\n'
        f'    <td>entities</td>\n'
        f'    <td>{entities}</td>\n'
        f'  </tr>\n'
        f'</table>\n'
    )
    wrapped_pages.append(
        f"<page-{page_no:03d}>\n{page_meta}{page.strip()}\n</page-{page_no:03d}>\n"
    )
final_md = "\n\n".join(wrapped_pages)

final_path = OUTPUT_DIR / f"{doc_name}_final.md"
# logo figure 블록 제거 (이미지 저장도 안 했으므로)
import re
final_md = re.sub(r'<table class="figure-meta">\n(?:(?!</table>).)*?<td>logo</td>.*?</table>', '', final_md, flags=re.DOTALL)
final_path.write_text(final_md, encoding="utf-8")
print(f"최종 마크다운 저장: {final_path} ({pic_idx}개 이미지, {tbl_idx}개 테이블, {len(pages)}페이지)")
print("\n--- 미리보기 (앞 500자) ---")
print(final_md[:500])


## 8. 결과 요약

In [ ]:
output_files = list(OUTPUT_DIR.iterdir())
print(f"출력 디렉토리: {OUTPUT_DIR.resolve()}")
print(f"생성된 파일 수: {len(output_files)}개\n")

for f in sorted(output_files):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:50s} {size_kb:8.1f} KB")